# 🚗 Ambulance Chaser — C4: Long-tail / Rare Scenario Mining

**Google Colab Notebook** — Chạy pipeline đầy đủ với CLIP thật trên GPU T4 miễn phí.

### Cách dùng:
1. Mở notebook này trên Colab: `Runtime → Change runtime type → T4 GPU`
2. Chạy từng cell từ trên xuống
3. Kết quả và biểu đồ sẽ hiện trực tiếp trong notebook

In [ ]:
# Cell 1: Cài đặt dependencies
!pip install -q open-clip-torch faiss-cpu scikit-learn matplotlib seaborn tabulate tqdm gdown

In [ ]:
# Cell 2: Download BDD100K (Solesensei dataset via Kaggle public link)
import os
import subprocess

DATA_DIR = "/content/bdd100k"

print("📥 Đang tải BDD100K dataset...")
!curl -L -o /content/solesensei_bdd100k.zip https://www.kaggle.com/api/v1/datasets/download/solesensei/solesensei_bdd100k

print("📦 Đang giải nén...")
!unzip -q /content/solesensei_bdd100k.zip -d /content/bdd100k
!rm /content/solesensei_bdd100k.zip

# Tìm thư mục chứa ảnh (do file zip giải nén có thể nằm trong thư mục con)
for root, dirs, files in os.walk('/content/bdd100k'):
    if len(files) > 100 and files[0].endswith('.jpg'):
        DATA_DIR = root
        break

print(f"✅ Dữ liệu đã sẵn sàng tại: {DATA_DIR}")


In [ ]:
# Cell 3: Import & Setup
import numpy as np
import torch
import open_clip
import faiss
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path
from tqdm.notebook import tqdm
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.manifold import TSNE
from tabulate import tabulate
from IPython.display import display, HTML, Markdown

np.random.seed(42)
torch.manual_seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 4: Load CLIP model
print("Loading CLIP ViT-B/32...")
model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='laion2b_s34b_b79k'
)
model = model.to(DEVICE).eval()
tokenizer = open_clip.get_tokenizer('ViT-B-32')
print("✓ CLIP loaded!")

In [ ]:
# Cell 5: Encode all images
data_dir = Path(DATA_DIR)
image_paths = sorted([p for p in data_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}])
print(f"Encoding {len(image_paths)} images...")

BATCH_SIZE = 64  # T4 can handle this
all_embeddings = []

with torch.no_grad():
    for i in tqdm(range(0, len(image_paths), BATCH_SIZE)):
        batch = image_paths[i:i+BATCH_SIZE]
        images = []
        for p in batch:
            try:
                images.append(preprocess(Image.open(p).convert('RGB')))
            except:
                images.append(preprocess(Image.new('RGB', (224, 224))))
        images = torch.stack(images).to(DEVICE)
        features = model.encode_image(images)
        features = features / features.norm(dim=-1, keepdim=True)
        all_embeddings.append(features.cpu().numpy())

all_embeddings = np.vstack(all_embeddings).astype(np.float32)
N = len(all_embeddings)
print(f"✓ Encoded {N} images → shape {all_embeddings.shape}")

In [ ]:
# Cell 6: Zero-shot scenario labeling
SCENARIO_PROMPTS = {
    'S1': ['overloaded cargo bike on road', 'three-wheeled vehicle with oversized load'],
    'S2': ['water buffalo on highway', 'livestock crossing the road', 'cows on the street'],
    'S3': ['funeral procession blocking road', 'street wedding tent on road', 'night market on street'],
    'S4': ['flooded road with vehicles', 'cars in deep flood water', 'motorcycle in flood'],
    'S5': ['ambulance in motorcycle traffic', 'emergency vehicle in dense traffic'],
    'S6': ['dark night road with person', 'construction at night on highway', 'poorly lit dangerous road'],
}

SCENARIO_LABELS = {
    'S1': '🚐 Xe quá khổ', 'S2': '🐃 Động vật', 'S3': '⛺ Sự kiện đường',
    'S4': '🌊 Ngập lụt', 'S5': '🚑 Xe khẩn cấp', 'S6': '🌙 Đêm compound',
}

# Encode scenario prompts
scenario_embs = {}
with torch.no_grad():
    for key, prompts in SCENARIO_PROMPTS.items():
        tokens = tokenizer(prompts).to(DEVICE)
        text_feat = model.encode_text(tokens)
        text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)
        scenario_embs[key] = text_feat.cpu().numpy().mean(axis=0)

# Score each image against each scenario
scenario_scores = {key: all_embeddings @ emb for key, emb in scenario_embs.items()}

# Mark rare: top 5% for each scenario
is_rare = np.zeros(N, dtype=bool)
group_ids = np.full(N, -1)

for gid, (key, scores) in enumerate(scenario_scores.items()):
    thresh = np.percentile(scores, 95)
    mask = scores >= thresh
    is_rare |= mask
    for idx in np.where(mask & (group_ids == -1))[0]:
        group_ids[idx] = gid

n_rare = is_rare.sum()
print(f"✓ Found {n_rare} rare frames ({n_rare/N:.1%})")
for gid, key in enumerate(SCENARIO_PROMPTS):
    c = (group_ids == gid).sum()
    print(f"   {SCENARIO_LABELS[key]}: {c}")

In [ ]:
# Cell 7: AMBULANCE CHASER PIPELINE
K_NN = 50
BUDGET = 0.05
ALPHA, BETA, GAMMA = 0.40, 0.35, 0.25
STRIDE = 30
eps = 1e-8
norm = lambda x: (x - x.min()) / (x.max() - x.min() + eps)

# 1) k-NN Density
print("1) k-NN density...")
index = faiss.IndexFlatIP(all_embeddings.shape[1])
index.add(all_embeddings)
dists, _ = index.search(all_embeddings, K_NN + 1)
novelty = 1.0 - dists[:, -1]

# 2) Ensemble Uncertainty
print("2) Ensemble uncertainty...")
n_train = min(2000, N // 3)
ti = np.random.choice(N, n_train, replace=False)
clfs = [
    RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300, random_state=42),
    GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42),
]
probas = []
for clf in clfs:
    clf.fit(all_embeddings[ti], is_rare[ti].astype(int))
    probas.append(clf.predict_proba(all_embeddings)[:, 1])
uncertainty = np.std(probas, axis=0)

# 3) Temporal Diversity
print("3) Temporal diversity...")
temporal_div = np.ones(N)
for i in range(1, N):
    lb = max(0, i - STRIDE)
    sims = all_embeddings[lb:i] @ all_embeddings[i]
    temporal_div[i] = 1.0 - sims.max()

# 4) Compound Score
compound = ALPHA * norm(novelty) + BETA * norm(uncertainty) + GAMMA * norm(temporal_div)

# 5) Select with stride
budget_n = int(N * BUDGET)
sorted_idx = np.argsort(compound)[::-1]
selected = []
for idx in sorted_idx:
    if len(selected) >= budget_n: break
    if not any(abs(int(idx) - int(s)) < STRIDE for s in selected[-STRIDE:]):
        selected.append(idx)
our_sel = np.array(selected)

# Baselines
rand_sel = np.random.choice(N, budget_n, replace=False)
unc_sel = np.argsort(uncertainty)[-budget_n:]
div_sel = np.argsort(norm(novelty))[-budget_n:]

print(f"\n✓ Selected {len(our_sel)} frames @ {BUDGET:.0%} budget")

In [ ]:
# Cell 8: RESULTS TABLE
def recall(sel): return is_rare[sel].sum() / max(is_rare.sum(), 1)
def cov(sel): return len(set(group_ids[i] for i in sel if group_ids[i] >= 0))
def redun(sel):
    sub = np.random.choice(sel, min(1000, len(sel)), replace=False)
    s = all_embeddings[sub] @ all_embeddings[sub].T
    np.fill_diagonal(s, 0)
    return s.mean()

methods = {'Random': rand_sel, 'Uncertainty': unc_sel, 'Diversity': div_sel, 'Ours': our_sel}
rows = []
for name, sel in methods.items():
    r, c, rd = recall(sel), cov(sel), redun(sel)
    rows.append([name, f'{r:.1%}', f'{c}/6', f'{rd:.3f}'])

display(Markdown('### 📊 Kết quả so sánh @ Budget 5%'))
display(Markdown(tabulate(rows, headers=['Method', 'Rare Recall', 'Coverage', 'Redundancy'], tablefmt='github')))

In [ ]:
# Cell 9: VISUALIZATION
plt.style.use('dark_background')
BG = '#0a0e1a'

fig, axes = plt.subplots(1, 3, figsize=(20, 6), facecolor=BG)

# Chart 1: Comparison bars
ax = axes[0]
ax.set_facecolor(BG)
names = list(methods.keys())
recalls = [recall(sel) for sel in methods.values()]
colors = ['#64748b', '#f59e0b', '#3b82f6', '#10b981']
ax.barh(names, recalls, color=colors, height=0.6)
for i, v in enumerate(recalls):
    ax.text(v + 0.02, i, f'{v:.1%}', va='center', fontweight='bold', color='white')
ax.set_title('Rare-Case Recall @ 5%', fontweight='bold', color='white', fontsize=13)
ax.set_xlim(0, max(recalls) * 1.3)

# Chart 2: Score distributions
ax = axes[1]
ax.set_facecolor(BG)
ax.hist(compound[~is_rare], bins=50, alpha=0.5, color='#64748b', label='Normal', density=True)
ax.hist(compound[is_rare], bins=50, alpha=0.8, color='#10b981', label='Rare', density=True)
ax.set_title('Compound Score Distribution', fontweight='bold', color='white', fontsize=13)
ax.legend()

# Chart 3: Budget sweep
ax = axes[2]
ax.set_facecolor(BG)
budgets_test = [0.01, 0.03, 0.05, 0.10, 0.15, 0.20]
r_rand = [recall(np.random.choice(N, int(N*b), replace=False)) for b in budgets_test]
r_ours = [recall(np.argsort(compound)[-int(N*b):]) for b in budgets_test]
ax.plot([b*100 for b in budgets_test], r_rand, 'o--', color='#64748b', label='Random', linewidth=2)
ax.plot([b*100 for b in budgets_test], r_ours, 'o-', color='#10b981', label='Ours', linewidth=3)
ax.fill_between([b*100 for b in budgets_test], r_rand, r_ours, alpha=0.15, color='#10b981')
ax.set_xlabel('Budget (%)', color='white')
ax.set_ylabel('Rare-Case Recall', color='white')
ax.set_title('Recall vs Budget', fontweight='bold', color='white', fontsize=13)
ax.legend()
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('/content/results.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('\n✅ Đã lưu results.png')

In [ ]:
# Cell 10: t-SNE (optional — mất 2-3 phút)
print("Computing t-SNE...")
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
coords = tsne.fit_transform(all_embeddings)

fig, ax = plt.subplots(figsize=(12, 10), facecolor=BG)
ax.set_facecolor(BG)

sel_mask = np.zeros(N, dtype=bool)
sel_mask[our_sel] = True

ax.scatter(coords[~is_rare & ~sel_mask, 0], coords[~is_rare & ~sel_mask, 1], c='#334155', alpha=0.15, s=5)
ax.scatter(coords[sel_mask & ~is_rare, 0], coords[sel_mask & ~is_rare, 1], c='#3b82f6', alpha=0.5, s=15, label='Selected (normal)')
ax.scatter(coords[sel_mask & is_rare, 0], coords[sel_mask & is_rare, 1], c='#10b981', s=50, alpha=0.9, label='Selected (rare) ✓', edgecolors='white', linewidths=0.5)
ax.scatter(coords[is_rare & ~sel_mask, 0], coords[is_rare & ~sel_mask, 1], c='#ef4444', s=30, marker='x', alpha=0.7, label='Missed rare ✗')

ax.set_title('Ambulance Chaser — t-SNE Visualization', fontweight='bold', color='white', fontsize=16)
ax.legend(fontsize=11, facecolor='#1a2035')
plt.tight_layout()
plt.savefig('/content/tsne.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('\n✅ Đã lưu tsne.png')

In [ ]:
# Cell 11: Download kết quả về máy
from google.colab import files

# Pack all outputs
!zip -q /content/c4_results.zip /content/results.png /content/tsne.png
files.download('/content/c4_results.zip')
print('\n📦 Đã download c4_results.zip!')